# 🔀 Qwen3 + Solar Merge 앙상블

## 실행 순서
1. Cell 1~4: 환경 설정 & 데이터 로드
2. Cell 5~6: 결과 파일 로드 & ROUGE 비교
3. Cell 7~8: Merge 앙상블 실행
4. Cell 9: 제출 파일 생성


## 1. Import

In [ ]:
import os
import re
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rouge import Rouge
from mecab import MeCab
from openai import OpenAI

DATA_PATH   = "/data/ephemeral/home/data/"
RESULT_PATH = "/data/ephemeral/home/code/prediction/"

# Solar API
client = OpenAI(
    api_key="up_opnjqOc0T694rx1It5hDPhfZNCiW5",
    base_url="https://api.upstage.ai/v1/solar"
)

rouge = Rouge()
m     = MeCab()

def tok(text):
    tokens = [t for t, _ in m.pos(str(text)) if t.strip()]
    return " ".join(tokens) if tokens else str(text)

def rouge_avg(pred, gold):
    try:
        s = rouge.get_scores(tok(pred), tok(gold))[0]
        return (s["rouge-1"]["f"] + s["rouge-2"]["f"] + s["rouge-l"]["f"]) / 3
    except:
        return 0.0

print("설정 완료")


## 2. 데이터 로드

In [ ]:
dev_df  = pd.read_csv(os.path.join(DATA_PATH, "dev.csv"))
test_df = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print(f"Dev : {len(dev_df)}개")
print(f"Test: {len(test_df)}개")


## 3. 결과 파일 로드

⭐ 파일명 확인 후 수정하세요


In [ ]:
import glob

# 파일 목록 확인
print("[prediction 폴더 파일 목록]")
files = sorted(os.listdir(RESULT_PATH))
for f in files:
    print(f"  {f}")


In [ ]:
# ============================================================
# ⭐ 실제 파일명으로 수정하세요
# ============================================================

# Solar test 결과
SOLAR_TEST_PATH = os.path.join(RESULT_PATH, "output_solar_v7.csv")

# Qwen3 test 결과 (학습 완료 후 생성된 파일)
QWEN_TEST_PATH  = os.path.join(RESULT_PATH, "submission_qwen3_8b_retrieval_v2.csv")

# Solar dev 결과 (validate 결과)
SOLAR_DEV_PATH  = os.path.join(RESULT_PATH, "solar_val_results_v7.csv")

# Qwen3 dev 결과
QWEN_DEV_PATH   = os.path.join(RESULT_PATH, "submission_qwen3_dev.csv")  # ⭐ 수정

# 파일 존재 확인
for name, path in [("Solar test", SOLAR_TEST_PATH), ("Qwen3 test", QWEN_TEST_PATH),
                   ("Solar dev",  SOLAR_DEV_PATH),  ("Qwen3 dev",  QWEN_DEV_PATH)]:
    exists = "✅" if os.path.exists(path) else "❌ 없음"
    print(f"  {exists} {name}: {path}")


## 4. Dev ROUGE 비교 (Solar vs Qwen3)

In [ ]:
def eval_rouge_file(pred_path, pred_col="summary", label=""):
    """파일 로드 후 dev ROUGE 계산"""
    pred_df = pd.read_csv(pred_path).sort_values("fname").reset_index(drop=True)
    dev     = dev_df.sort_values("fname").reset_index(drop=True)

    preds = [str(p).strip() for p in pred_df[pred_col]]
    golds = [str(g).strip() for g in dev["summary"]]

    pt = [tok(p) for p in preds]
    gt = [tok(g) for g in golds]
    valid = [(p,g) for p,g in zip(pt,gt) if p.strip() and g.strip()]

    s   = rouge.get_scores([p for p,g in valid], [g for p,g in valid], avg=True)
    r1  = s["rouge-1"]["f"]
    r2  = s["rouge-2"]["f"]
    rl  = s["rouge-l"]["f"]
    avg = (r1+r2+rl)/3
    print(f"[{label:<15}] R1={r1:.4f}  R2={r2:.4f}  RL={rl:.4f}  AVG={avg:.4f}")
    return avg


print("="*60)
print("📊 Dev ROUGE 비교 (MeCab 기반)")
print("="*60)

scores = {}
try:
    scores["solar"] = eval_rouge_file(SOLAR_DEV_PATH, pred_col="pred", label="Solar v7")
except Exception as e:
    print(f"Solar dev 파일 오류: {e}")

try:
    scores["qwen3"] = eval_rouge_file(QWEN_DEV_PATH, label="Qwen3")
except Exception as e:
    print(f"Qwen3 dev 파일 오류: {e}")

print("="*60)
if scores:
    best = max(scores, key=scores.get)
    print(f"✅ 단독 최고: {best.upper()} (AVG={scores[best]:.4f})")


## 5. Merge 앙상블 함수

In [ ]:
def merge_ensemble(dialogue, qwen_summary, solar_summary):
    """
    Qwen3 + Solar → Solar에게 merge 요청
    
    로직:
    1. Solar가 이미 조건 만족 (≤20단어 + #Person 태그) → 그대로 사용
    2. 그 외 → Solar API로 merge
    3. merge 결과에 #Person 없으면 → solar_summary 사용
    4. API 실패 → qwen_summary 사용
    """
    # ✅ Solar가 이미 좋은 경우 그대로 사용
    solar_len = len(str(solar_summary).split())
    if solar_len <= 20 and "#Person" in str(solar_summary):
        return solar_summary

    merge_prompt = [
        {
            "role": "system",
            "content": (
                "두 요약문을 참고하여 가장 정확한 최종 요약 1문장을 작성하세요.\n"
                "규칙: 1문장, 20단어 이하, #Person 태그 유지, 핵심 사건만 포함"
            )
        },
        {
            "role": "user",
            "content": (
                f"대화:\n{dialogue}\n\n"
                f"요약 후보1: {qwen_summary}\n"
                f"요약 후보2: {solar_summary}\n\n"
                "두 요약을 참고하여 핵심만 담은 최종 요약 1문장:"
            )
        }
    ]

    try:
        response = client.chat.completions.create(
            model="solar-1-mini-chat",
            messages=merge_prompt,
            temperature=0.1,
            max_tokens=60,
        )
        result = response.choices[0].message.content.strip()

        # 길이 후처리
        words = result.split()
        if len(words) > 20:
            result = " ".join(words[:20])

        # ✅ #Person 태그 없으면 solar로 폴백
        if "#Person" not in result:
            return solar_summary

        return result

    except:
        return qwen_summary   # API 실패 → Qwen3 사용


print("merge_ensemble 함수 정의 완료")


## 6. Dev Merge ROUGE 비교 (선택사항 - 시간 여유 있을 때)

In [ ]:
# ⭐ 시간 없으면 건너뛰고 Cell 7로 바로 이동

try:
    solar_dev = pd.read_csv(SOLAR_DEV_PATH).sort_values("fname").reset_index(drop=True)
    qwen_dev  = pd.read_csv(QWEN_DEV_PATH).sort_values("fname").reset_index(drop=True)
    dev_sorted = dev_df.sort_values("fname").reset_index(drop=True)

    solar_preds_dev = [str(p).strip() for p in solar_dev["pred"]]
    qwen_preds_dev  = [str(p).strip() for p in qwen_dev["summary"]]
    golds_dev       = [str(g).strip() for g in dev_sorted["summary"]]

    print("Dev Merge 추론 시작 (50개)...")
    merge_preds_dev = []
    for i in tqdm(range(min(50, len(dev_sorted)))):
        merged = merge_ensemble(
            dev_sorted.iloc[i]["dialogue"],
            qwen_preds_dev[i],
            solar_preds_dev[i]
        )
        merge_preds_dev.append(merged)

    golds_50 = golds_dev[:50]
    print("\n" + "="*60)
    print("📊 Dev Merge 비교 (50개 샘플)")
    print("="*60)

    for label, preds in [("Solar", solar_preds_dev[:50]),
                          ("Qwen3", qwen_preds_dev[:50]),
                          ("Merge", merge_preds_dev)]:
        pt = [tok(p) for p in preds]
        gt = [tok(g) for g in golds_50]
        valid = [(p,g) for p,g in zip(pt,gt) if p.strip() and g.strip()]
        s   = rouge.get_scores([p for p,g in valid],[g for p,g in valid],avg=True)
        avg = (s["rouge-1"]["f"]+s["rouge-2"]["f"]+s["rouge-l"]["f"])/3
        print(f"  [{label:<6}] AVG={avg:.4f}")
    print("="*60)

except Exception as e:
    print(f"건너뜀: {e}")


## 7. Test Merge 앙상블 실행

In [ ]:
# 결과 파일 로드
solar_test = pd.read_csv(SOLAR_TEST_PATH).sort_values("fname").reset_index(drop=True)
qwen_test  = pd.read_csv(QWEN_TEST_PATH).sort_values("fname").reset_index(drop=True)
test_sorted = test_df.sort_values("fname").reset_index(drop=True)

# fname 일치 확인
assert list(solar_test["fname"]) == list(qwen_test["fname"]), "❌ fname 불일치!"
print(f"✅ fname 일치 확인 ({len(solar_test)}개)")

solar_preds = [str(p).strip() for p in solar_test["summary"]]
qwen_preds  = [str(p).strip() for p in qwen_test["summary"]]


In [ ]:
print(f"Merge 앙상블 시작 ({len(test_sorted)}개)...")
final_preds = []
merge_count = 0   # 실제 merge된 횟수
solar_count = 0   # Solar 그대로 사용 횟수

start_time = time.time()

for i in tqdm(range(len(test_sorted))):
    dialogue = test_sorted.iloc[i]["dialogue"]
    q = qwen_preds[i]
    s = solar_preds[i]

    # Solar 조건 만족 시 API 호출 없이 바로 사용
    solar_len = len(str(s).split())
    if solar_len <= 20 and "#Person" in str(s):
        final_preds.append(s)
        solar_count += 1
    else:
        merged = merge_ensemble(dialogue, q, s)
        final_preds.append(merged)
        merge_count += 1

    # RPM 제한 방지 (merge 호출 60개마다 대기)
    if merge_count > 0 and merge_count % 60 == 0:
        elapsed = time.time() - start_time
        if elapsed < 60:
            wait = 60 - elapsed + 3
            print(f"  RPM 대기: {wait:.0f}초")
            time.sleep(wait)
        start_time = time.time()

    if i % 50 == 0:
        print(f"  [{i}/{len(test_sorted)}] Solar 그대로: {solar_count} | Merge: {merge_count}")

print(f"\n✅ 앙상블 완료!")
print(f"  Solar 그대로 사용: {solar_count}개 ({solar_count/len(test_sorted)*100:.1f}%)")
print(f"  실제 Merge 호출  : {merge_count}개 ({merge_count/len(test_sorted)*100:.1f}%)")


## 8. 제출 파일 생성

In [ ]:
submission = pd.DataFrame({
    "fname"  : test_sorted["fname"],
    "summary": final_preds,
})

save_path = os.path.join(RESULT_PATH, "submission_merge_ensemble.csv")
submission.to_csv(save_path, index=False)

print(f"✅ 제출 파일 저장: {save_path}")
print(f"총 {len(submission)}개 | 고유 요약: {submission['summary'].nunique()}개")
print(f"평균 길이  : {submission['summary'].str.split().str.len().mean():.1f}단어")
print(f"#Person 포함: {submission['summary'].str.contains('#Person').mean():.1%}")
print(f"\n[미리보기]")
print(submission.head(10).to_string())


## 9. 최종 비교 요약

In [ ]:
print("="*55)
print("📊 제출 파일 비교")
print("="*55)
print(f"Solar 단독   : {SOLAR_TEST_PATH.split('/')[-1]}")
print(f"Qwen3 단독   : {QWEN_TEST_PATH.split('/')[-1]}")
print(f"Merge 앙상블 : submission_merge_ensemble.csv")
print("="*55)
print("\n⭐ 제출 우선순위 판단:")
print("  1. Dev ROUGE 가장 높은 것 제출")
print("  2. 시간 없으면 Solar(0.4894) or Merge 중 선택")
